In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        print(os.path.join(root, file))

/content/qa_data.json
/content/.config/gce
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/default_configs.db
/content/.config/.last_update_check.json
/content/.config/.last_opt_in_prompt.yaml
/content/.config/config_sentinel
/content/.config/active_config
/content/.config/.last_survey_prompt.yaml
/content/.config/logs/2026.10.21/13.27.57.480371.log
/content/.config/logs/2026.10.21/13.28.04.564766.log
/content/.config/logs/2026.10.21/13.27.58.650478.log
/content/.config/logs/2026.10.21/13.28.05.116687.log
/content/.config/logs/2026.10.21/13.27.51.774504.log
/content/.config/logs/2026.10.21/13.27.28.750982.log
/content/.config/configurations/config_default
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_json(
    "/content/qa_data.json",
    lines=True
)

# Dataset information
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
display(df.head())

Dataset Shape: (10000, 4)

Column Names:
['knowledge', 'question', 'right_answer', 'hallucinated_answer']

First 5 Records:


,knowledge,question,right_answer,hallucinated_answer
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,First for Women was started first.
1,The Oberoi family is an Indian family that is ...,The Oberoi family is part of a hotel company t...,Delhi,The Oberoi family's hotel company is based in ...
2,"Allison Beth ""Allie"" Goertz (born March 2, 199...",Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,"Allie Goertz wrote a song about Milhouse, a po..."
3,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",What nationality was James Henry Miller's wife?,American,James Henry Miller's wife was British.
4,It is a hygroscopic solid that is highly solu...,Cadmium Chloride is slightly soluble in this c...,alcohol,water with a hint of alcohol


In [ ]:
# Missing values
print("Missing Values:")
print(df.isnull().sum())

# Duplicate records
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Dataset information
print("\nDataset Info:")
df.info()

Missing Values:
knowledge              0
question               0
right_answer           0
hallucinated_answer    0
dtype: int64

Duplicate Rows:
0

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   knowledge            10000 non-null  object
 1   question             10000 non-null  object
 2   right_answer         10000 non-null  object
 3   hallucinated_answer  10000 non-null  object
dtypes: object(4)
memory usage: 312.6+ KB


In [ ]:
# Select 100 QA pairs
small_df = df.sample(
    n=100,
    random_state=42
).reset_index(drop=True)

# Save selected data
small_df.to_json(
    "/content/small_dataset.json",
    orient="records",
    lines=True
)

print("Small Dataset Shape:", small_df.shape)

Small Dataset Shape: (100, 4)


In [ ]:
# Right answers
right_df = small_df[
    ['knowledge', 'question', 'right_answer']
].copy()

right_df = right_df.rename(
    columns={'right_answer': 'claim'}
)

right_df['label'] = 0
right_df['claim_type'] = 'right_answer'


# Hallucinated answers
hallucinated_df = small_df[
    ['knowledge', 'question', 'hallucinated_answer']
].copy()

hallucinated_df = hallucinated_df.rename(
    columns={'hallucinated_answer': 'claim'}
)

hallucinated_df['label'] = 1
hallucinated_df['claim_type'] = 'hallucinated_answer'


# Combine both datasets
claims_df = pd.concat(
    [right_df, hallucinated_df],
    ignore_index=True
)

# Shuffle the rows
claims_df = claims_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# Save the dataset
claims_df.to_csv(
    '/content/claims_dataset.csv',
    index=False
)


print("New Dataset Shape:", claims_df.shape)
print("\nLabel Distribution:")
print(claims_df['label'].value_counts())

display(claims_df.head())

New Dataset Shape: (200, 5)

Label Distribution:
label
0    100
1    100
Name: count, dtype: int64


,knowledge,question,claim,label,claim_type
0,Danyang () is a county-level city under the ad...,"Are Danyang, Jiangsu and Dongfang, Hainan both...",yes,0,right_answer
1,The Idaho Vandals are the college football tea...,What is the name of the American football coac...,Paul Petrino,0,right_answer
2,"The Mustangs, led by third-year head coach Ju...",Which league does the 2010 head coach of the S...,Canadian Football League,0,right_answer
3,Battletoads/Double Dragon is a 1993 beat 'em u...,Battletoads/Double Dragon is a game that can b...,You can play Battletoads/Double Dragon on the ...,1,hallucinated_answer
4,"Juliet Snowden is an American screenwriter, fi...",which film dirtected by Alex Proyas and starri...,Juliet Snowden co-directed Knowing with Alex P...,1,hallucinated_answer


In [ ]:
# Create a common group ID for paired claims

claims_df['group_id'] = (
    claims_df.groupby(['knowledge', 'question']).ngroup()
)

# Check number of groups
print("Total QA Groups:", claims_df['group_id'].nunique())

# Check claims in each group
print("\nClaims per Group:")
print(claims_df.groupby('group_id').size().value_counts())

# Check labels in first 5 groups
print("\nFirst 5 Groups:")
display(
    claims_df[
        claims_df['group_id'].isin([0, 1, 2, 3, 4])
    ][['group_id', 'question', 'claim', 'label']]
)

Total QA Groups: 100

Claims per Group:
2    100
Name: count, dtype: int64

First 5 Groups:


,group_id,question,claim,label
25,1,What is the name of the baseball park located ...,Target Field,0
56,0,Marc Abaya appeared in the 2009 Phillippine re...,Marc Abaya appeared in the 2009 Phillippine re...,1
58,3,The Toyota TF102 was the car with which the To...,"As with the TF101, the Toyota TF102 was pilote...",1
105,2,What nationality is the player who scored afte...,"Louis Saha scored in the 2009 FA Cup Final, af...",1
106,4,Which player singed to Bayern Munich was born ...,Dante Bonfim Costa Santos (born 18 October 1983),0
117,2,What nationality is the player who scored afte...,French,0
120,0,Marc Abaya appeared in the 2009 Phillippine re...,DisneyToon Studios,0
147,4,Which player singed to Bayern Munich was born ...,The player signed by Bayern Munich who was bor...,1
159,3,The Toyota TF102 was the car with which the To...,Allan McNish,0
174,1,What is the name of the baseball park located ...,Minnesota Ballpark in North Loop.,1


In [ ]:
!pip install -q transformers==4.57.1 sentencepiece accelerate

In [ ]:
import torch
from transformers import pipeline

model_name = "google/flan-t5-base"

device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text2text-generation",
    model=model_name,
    device=device
)

print("Model loaded successfully!")
print("Device:", "GPU" if device == 0 else "CPU")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Model loaded successfully!
Device: GPU


In [ ]:
probes = {
    "evidence": (
        "Using only the supplied evidence, answer the question. "
        "If the claim is unsupported, state what the evidence supports."
    ),

    "contradiction": (
        "Check whether the claim is contradicted by the evidence. "
        "If so, state the evidence-supported answer."
    ),

    "temporal": (
        "Check dates, chronology, and event order in the evidence. "
        "Answer only what is temporally supported."
    ),

    "source": (
        "Identify the exact evidence statement supporting the answer. "
        "If none exists, say the evidence does not support it."
    )
}

print("Total Probes:", len(probes))
print(list(probes.keys()))

Total Probes: 4
['evidence', 'contradiction', 'temporal', 'source']


In [ ]:
import pandas as pd

# Reload claims_df since the runtime memory was reset
claims_df = pd.read_csv('/content/claims_dataset.csv')

# Recreate group_id since it was not saved in the CSV file
claims_df['group_id'] = (
    claims_df.groupby(['knowledge', 'question']).ngroup()
)

test_df = claims_df.head(2).copy()

display(
    test_df[['group_id', 'question', 'claim', 'label']]
)

,group_id,question,claim,label
0,52,"Are Danyang, Jiangsu and Dongfang, Hainan both...",yes,0
1,87,What is the name of the American football coac...,Paul Petrino,0


In [ ]:
test_df = (
    claims_df.groupby('group_id')
    .head(2)
    .groupby('label', group_keys=False)
    .head(1)
)

test_df = test_df.groupby('group_id').head(2).copy()

display(
    test_df[['group_id', 'question', 'claim', 'label']]
)

,group_id,question,claim,label
0,52,"Are Danyang, Jiangsu and Dongfang, Hainan both...",yes,0
3,46,Battletoads/Double Dragon is a game that can b...,You can play Battletoads/Double Dragon on the ...,1


In [ ]:
test_df = claims_df[
    claims_df['group_id'] == claims_df['group_id'].iloc[0]
].copy()

display(
    test_df[['group_id', 'question', 'claim', 'label']]
)

,group_id,question,claim,label
0,52,"Are Danyang, Jiangsu and Dongfang, Hainan both...",yes,0
45,52,"Are Danyang, Jiangsu and Dongfang, Hainan both...","Danyang and Dongfang are cities, but not in Ch...",1


In [ ]:
probes = {
    "evidence": (
        "Using only the supplied evidence, answer the question. "
        "If the claim is unsupported, state what the evidence supports."
    ),

    "contradiction": (
        "Check whether the claim is contradicted by the evidence. "
        "If so, state the evidence-supported answer."
    ),

    "temporal": (
        "Check dates, chronology, and event order in the evidence. "
        "Answer only what is temporally supported."
    ),

    "source": (
        "Identify the exact evidence statement supporting the answer. "
        "If none exists, say the evidence does not support it."
    )
}

print("Total Probes:", len(probes))

Total Probes: 4


In [ ]:
results = []

for _, row in test_df.iterrows():

    for probe_name, instruction in probes.items():

        prompt = f"""
{instruction}

Question: {row['question']}

Candidate claim: {row['claim']}

Evidence:
{row['knowledge']}
"""

        output = generator(
            prompt,
            max_new_tokens=100,
            do_sample=False,
            num_beams=1
        )

        results.append({
            "group_id": row['group_id'],
            "claim": row['claim'],
            "label": row['label'],
            "probe": probe_name,
            "response": output[0]['generated_text']
        })

probe_test_df = pd.DataFrame(results)

display(probe_test_df)

,group_id,claim,label,probe,response
0,52,yes,0,evidence,yes
1,52,yes,0,contradiction,yes
2,52,yes,0,temporal,yes
3,52,yes,0,source,yes
4,52,"Danyang and Dongfang are cities, but not in Ch...",1,evidence,no
5,52,"Danyang and Dongfang are cities, but not in Ch...",1,contradiction,no
6,52,"Danyang and Dongfang are cities, but not in Ch...",1,temporal,no
7,52,"Danyang and Dongfang are cities, but not in Ch...",1,source,no


In [ ]:
print("Total Responses:", len(probe_test_df))

print("\nResponses per Probe:")
print(probe_test_df['probe'].value_counts())

Total Responses: 8

Responses per Probe:
probe
evidence         2
contradiction    2
temporal         2
source           2
Name: count, dtype: int64


In [ ]:
pd.set_option('display.max_colwidth', None)

display(
    probe_test_df[
        ['claim', 'label', 'probe', 'response']
    ].sort_values(['label', 'probe'])
)

,claim,label,probe,response
1,yes,0,contradiction,yes
0,yes,0,evidence,yes
3,yes,0,source,yes
2,yes,0,temporal,yes
5,"Danyang and Dongfang are cities, but not in China.",1,contradiction,no
4,"Danyang and Dongfang are cities, but not in China.",1,evidence,no
7,"Danyang and Dongfang are cities, but not in China.",1,source,no
6,"Danyang and Dongfang are cities, but not in China.",1,temporal,no


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("Embedding Model Loaded Successfully!")

Embedding Model Loaded Successfully!


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Question যুক্ত করে claim-এর text তৈরি
probe_test_df = probe_test_df.merge(
    test_df[['group_id', 'claim', 'question']],
    on=['group_id', 'claim'],
    how='left'
)

probe_test_df['claim_text'] = (
    "Question: " + probe_test_df['question'] +
    " Answer: " + probe_test_df['claim']
)

# Claim ও response-এর embedding তৈরি
claim_embeddings = embedding_model.encode(
    probe_test_df['claim_text'].tolist(),
    convert_to_numpy=True
)

response_embeddings = embedding_model.encode(
    probe_test_df['response'].tolist(),
    convert_to_numpy=True
)

# প্রতিটি claim ও তার response-এর cosine similarity
similarities = np.diag(
    cosine_similarity(claim_embeddings, response_embeddings)
)

# Semantic drift = 1 - cosine similarity
probe_test_df['cosine_similarity'] = similarities
probe_test_df['semantic_drift'] = 1 - similarities

print("Claim Embedding Shape:", claim_embeddings.shape)
print("Response Embedding Shape:", response_embeddings.shape)

display(
    probe_test_df[
        ['label', 'probe', 'claim', 'response',
         'cosine_similarity', 'semantic_drift']
    ]
)

Claim Embedding Shape: (8, 384)
Response Embedding Shape: (8, 384)


,label,probe,claim,response,cosine_similarity,semantic_drift
0,0,evidence,yes,yes,0.144517,0.855483
1,0,contradiction,yes,yes,0.144517,0.855483
2,0,temporal,yes,yes,0.144517,0.855483
3,0,source,yes,yes,0.144517,0.855483
4,1,evidence,"Danyang and Dongfang are cities, but not in China.",no,0.090702,0.909298
5,1,contradiction,"Danyang and Dongfang are cities, but not in China.",no,0.090702,0.909298
6,1,temporal,"Danyang and Dongfang are cities, but not in China.",no,0.090702,0.909298
7,1,source,"Danyang and Dongfang are cities, but not in China.",no,0.090702,0.909298


In [ ]:
display(
    probe_test_df[
        ['question', 'claim_text', 'response']
    ].head(2)
)

,question,claim_text,response
0,"Are Danyang, Jiangsu and Dongfang, Hainan both cities in China?","Question: Are Danyang, Jiangsu and Dongfang, Hainan both cities in China? Answer: yes",yes
1,"Are Danyang, Jiangsu and Dongfang, Hainan both cities in China?","Question: Are Danyang, Jiangsu and Dongfang, Hainan both cities in China? Answer: yes",yes


In [ ]:
probe_test_df['response_with_context'] = (
    "Question: " + probe_test_df['question'] +
    " Answer: " + probe_test_df['response']
)

context_response_embeddings = embedding_model.encode(
    probe_test_df['response_with_context'].tolist(),
    convert_to_numpy=True
)

context_similarities = np.diag(
    cosine_similarity(
        claim_embeddings,
        context_response_embeddings
    )
)

probe_test_df['context_similarity'] = context_similarities
probe_test_df['context_drift'] = 1 - context_similarities

display(
    probe_test_df[
        ['label', 'response', 'cosine_similarity',
         'context_similarity', 'semantic_drift',
         'context_drift']
    ]
)

,label,response,cosine_similarity,context_similarity,semantic_drift,context_drift
0,0,yes,0.144517,1.000000,0.855483,5.960464e-08
1,0,yes,0.144517,1.000000,0.855483,5.960464e-08
2,0,yes,0.144517,1.000000,0.855483,5.960464e-08
3,0,yes,0.144517,1.000000,0.855483,5.960464e-08
4,1,no,0.090702,0.948514,0.909298,5.148613e-02
5,1,no,0.090702,0.948514,0.909298,5.148613e-02
6,1,no,0.090702,0.948514,0.909298,5.148613e-02
7,1,no,0.090702,0.948514,0.909298,5.148613e-02


In [ ]:
import pandas as pd

# Randomly select 10 QA groups
selected_groups = (
    claims_df['group_id']
    .drop_duplicates()
    .sample(n=10, random_state=42)
    .tolist()
)

# Keep both claims from each selected group
pilot_df = claims_df[
    claims_df['group_id'].isin(selected_groups)
].copy()

pilot_df = pilot_df.reset_index(drop=True)

print("Selected QA Groups:", len(selected_groups))
print("Total Claims:", len(pilot_df))

print("\nLabel Distribution:")
print(pilot_df['label'].value_counts())

display(
    pilot_df[
        ['group_id', 'question', 'claim', 'label']
    ].head(6)
)

Selected QA Groups: 10
Total Claims: 20

Label Distribution:
label
0    10
1    10
Name: count, dtype: int64


,group_id,question,claim,label
0,52,"Are Danyang, Jiangsu and Dongfang, Hainan both cities in China?",yes,0
1,5,"Which actress who starred in ""About Last Night..."" endorsed the Werewolf Diet?","The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",1
2,5,"Which actress who starred in ""About Last Night..."" endorsed the Werewolf Diet?",Demi Moore,0
3,1,What is the name of the baseball park located in the historic warehouse distric of downtown Minneapolis and is home of the Minnesota Twins?,Target Field,0
4,85,The Black Candle is a documentary film about a celebration created by who?,Maulana Karenga,0
5,52,"Are Danyang, Jiangsu and Dongfang, Hainan both cities in China?","Danyang and Dongfang are cities, but not in China.",1


In [ ]:
probes = {
    "evidence": (
        "Using only the supplied evidence, answer the question. "
        "If the claim is unsupported, state what the evidence supports."
    ),

    "contradiction": (
        "Check whether the claim is contradicted by the evidence. "
        "If so, state the evidence-supported answer."
    ),

    "temporal": (
        "Check dates, chronology, and event order in the evidence. "
        "Answer only what is temporally supported."
    ),

    "source": (
        "Identify the exact evidence statement supporting the answer. "
        "If none exists, say the evidence does not support it."
    )
}

results = []

for _, row in pilot_df.iterrows():

    for probe_name, instruction in probes.items():

        prompt = f"""
{instruction}

Question: {row['question']}

Candidate claim: {row['claim']}

Evidence:
{row['knowledge']}
"""

        output = generator(
            prompt,
            max_new_tokens=100,
            do_sample=False,
            num_beams=1
        )

        response = output[0]['generated_text'].strip()

        results.append({
            'group_id': row['group_id'],
            'label': row['label'],
            'claim': row['claim'],
            'probe': probe_name,
            'response': response
        })

probe_results_df = pd.DataFrame(results)

print("Total Probe Responses:", len(probe_results_df))

display(probe_results_df.head(12))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Total Probe Responses: 80


,group_id,label,claim,probe,response
0,52,0,yes,evidence,yes
1,52,0,yes,contradiction,yes
2,52,0,yes,temporal,yes
3,52,0,yes,source,yes
4,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",evidence,Angelina Jolie
5,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",contradiction,Yes
6,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",temporal,Angelina Jolie
7,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",source,Angelina Jolie
8,5,0,Demi Moore,evidence,Demi Moore
9,5,0,Demi Moore,contradiction,Demi Moore


In [ ]:
probe_results_df['normalized_response'] = (
    probe_results_df['response']
    .str.lower()
    .str.strip()
)

redundancy_check = (
    probe_results_df
    .groupby(['group_id', 'claim', 'label'])
    ['normalized_response']
    .nunique()
    .reset_index(name='unique_responses')
)

print("Response Redundancy:")
print(redundancy_check['unique_responses'].value_counts())

display(redundancy_check)

Response Redundancy:
unique_responses
1    14
2     3
4     2
3     1
Name: count, dtype: int64


,group_id,claim,label,unique_responses
0,1,Minnesota Ballpark in North Loop.,1,1
1,1,Target Field,0,1
2,4,Dante Bonfim Costa Santos (born 18 October 1983),0,1
3,4,The player signed by Bayern Munich who was born in 1983 is Dante.,1,2
4,5,Demi Moore,0,1
5,5,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",1,2
6,6,American,0,1
7,6,"Milton Friedman, the person who said Savage was a genius, was American.",1,1
8,8,335 mph,0,1
9,8,Antron Brown has won three championships but the speed he reaches is unknown.,1,1


In [ ]:
# প্রতিটি QA Group-এর দুইটি claim-এর redundancy পরীক্ষা
group_summary = (
    redundancy_check
    .groupby('group_id')
    .agg(
        total_claims=('claim', 'count'),
        both_claims_redundant=(
            'unique_responses',
            lambda x: (x == 1).all()
        ),
        nonredundant_claims=(
            'unique_responses',
            lambda x: (x > 1).sum()
        )
    )
    .reset_index()
)

print("Total QA Groups:", len(group_summary))

print("\nBoth claims fully redundant:")
print(group_summary['both_claims_redundant'].sum())

print("\nGroups with at least one non-redundant claim:")
print((~group_summary['both_claims_redundant']).sum())

display(group_summary)

Total QA Groups: 10

Both claims fully redundant:
6

Groups with at least one non-redundant claim:
4


,group_id,total_claims,both_claims_redundant,nonredundant_claims
0,1,2,True,0
1,4,2,False,1
2,5,2,False,1
3,6,2,True,0
4,8,2,True,0
5,25,2,True,0
6,52,2,True,0
7,57,2,False,2
8,85,2,True,0
9,95,2,False,2


In [ ]:
# যেসব claim-এ একাধিক response পাওয়া গেছে
nonredundant_claims = redundancy_check[
    redundancy_check['unique_responses'] > 1
][['group_id', 'claim', 'label']]

# তাদের চারটি probe-এর response দেখানো
different_responses = probe_results_df.merge(
    nonredundant_claims,
    on=['group_id', 'claim', 'label'],
    how='inner'
)

display(
    different_responses[
        ['group_id', 'label', 'claim', 'probe', 'response']
    ].sort_values(['group_id', 'label', 'probe'])
)

,group_id,label,claim,probe,response
13,4,1,The player signed by Bayern Munich who was born in 1983 is Dante.,contradiction,Dante
12,4,1,The player signed by Bayern Munich who was born in 1983 is Dante.,evidence,Dante Bonfim Costa Santos
15,4,1,The player signed by Bayern Munich who was born in 1983 is Dante.,source,Dante Bonfim Costa Santos
14,4,1,The player signed by Bayern Munich who was born in 1983 is Dante.,temporal,Dante Bonfim Costa Santos
1,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",contradiction,Yes
0,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",evidence,Angelina Jolie
3,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",source,Angelina Jolie
2,5,1,"The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie.",temporal,Angelina Jolie
21,57,0,film,contradiction,Yes
20,57,0,film,evidence,film


In [ ]:
import re

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [s.strip() for s in sentences if s.strip()]


graph_units = []

for _, row in pilot_df.iterrows():

    group_id = row['group_id']
    claim = row['claim']
    label = row['label']
    question = row['question']
    knowledge = row['knowledge']

    # Original claim unit
    claim_text = (
        "Question: " + question +
        " Answer: " + claim
    )

    # G0: Baseline units
    graph_units.append({
        'group_id': group_id,
        'label': label,
        'graph': 'G0',
        'unit_type': 'claim',
        'text': claim_text
    })

    # Knowledge sentences
    for sentence in split_sentences(knowledge):

        graph_units.append({
            'group_id': group_id,
            'label': label,
            'graph': 'G0',
            'unit_type': 'knowledge',
            'text': sentence
        })

    # G1: Same baseline units + four probe responses
    for _, probe_row in probe_results_df[
        (probe_results_df['group_id'] == group_id) &
        (probe_results_df['claim'] == claim)
    ].iterrows():

        graph_units.append({
            'group_id': group_id,
            'label': label,
            'graph': 'G1',
            'unit_type': 'probe_response',
            'text': probe_row['response']
        })


graph_units_df = pd.DataFrame(graph_units)

print("Total Text Units:", len(graph_units_df))

print("\nUnits by Graph:")
print(graph_units_df['graph'].value_counts())

print("\nUnits by Type:")
print(graph_units_df['unit_type'].value_counts())

display(graph_units_df.head(15))

Total Text Units: 130

Units by Graph:
graph
G1    80
G0    50
Name: count, dtype: int64

Units by Type:
unit_type
probe_response    80
knowledge         30
claim             20
Name: count, dtype: int64


,group_id,label,graph,unit_type,text
0,52,0,G0,claim,"Question: Are Danyang, Jiangsu and Dongfang, Hainan both cities in China? Answer: yes"
1,52,0,G0,knowledge,"Danyang () is a county-level city under the administration of Zhenjiang, Jiangsu province, China.Dōngfāng () is one of the seven county-level cities of Hainan province, China."
2,52,0,G1,probe_response,yes
3,52,0,G1,probe_response,yes
4,52,0,G1,probe_response,yes
5,52,0,G1,probe_response,yes
6,5,1,G0,claim,"Question: Which actress who starred in ""About Last Night..."" endorsed the Werewolf Diet? Answer: The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie."
7,5,1,G0,knowledge,Celebrities that have endorsed the diet include Demi Moore and Madonna.
8,5,1,G0,knowledge,"Her first film to become both a critical and commercial hit was ""About Last Night..."" (1986), which established her as a Hollywood star."
9,5,1,G1,probe_response,Angelina Jolie


In [ ]:
graph_units = []

for _, row in pilot_df.iterrows():

    group_id = row['group_id']
    claim = row['claim']
    label = row['label']
    question = row['question']
    knowledge = row['knowledge']

    claim_text = (
        "Question: " + question +
        " Answer: " + claim
    )

    # Baseline units: included in BOTH G0 and G1
    baseline_units = [
        {
            'unit_type': 'claim',
            'text': claim_text
        }
    ]

    for sentence in split_sentences(knowledge):
        baseline_units.append({
            'unit_type': 'knowledge',
            'text': sentence
        })

    # Add baseline units to G0 and G1
    for graph_name in ['G0', 'G1']:

        for unit in baseline_units:
            graph_units.append({
                'group_id': group_id,
                'label': label,
                'graph': graph_name,
                'unit_type': unit['unit_type'],
                'text': unit['text']
            })

    # Add probe responses ONLY to G1
    selected_responses = probe_results_df[
        (probe_results_df['group_id'] == group_id) &
        (probe_results_df['claim'] == claim)
    ]

    for _, probe_row in selected_responses.iterrows():

        graph_units.append({
            'group_id': group_id,
            'label': label,
            'graph': 'G1',
            'unit_type': 'probe_response',
            'text': probe_row['response']
        })


graph_units_df = pd.DataFrame(graph_units)

print("Total Text Units:", len(graph_units_df))

print("\nUnits by Graph:")
print(graph_units_df['graph'].value_counts())

print("\nUnits by Type and Graph:")
print(
    graph_units_df.groupby(
        ['graph', 'unit_type']
    ).size()
)

display(graph_units_df.head(12))

Total Text Units: 180

Units by Graph:
graph
G1    130
G0     50
Name: count, dtype: int64

Units by Type and Graph:
graph  unit_type     
G0     claim             20
       knowledge         30
G1     claim             20
       knowledge         30
       probe_response    80
dtype: int64


,group_id,label,graph,unit_type,text
0,52,0,G0,claim,"Question: Are Danyang, Jiangsu and Dongfang, Hainan both cities in China? Answer: yes"
1,52,0,G0,knowledge,"Danyang () is a county-level city under the administration of Zhenjiang, Jiangsu province, China.Dōngfāng () is one of the seven county-level cities of Hainan province, China."
2,52,0,G1,claim,"Question: Are Danyang, Jiangsu and Dongfang, Hainan both cities in China? Answer: yes"
3,52,0,G1,knowledge,"Danyang () is a county-level city under the administration of Zhenjiang, Jiangsu province, China.Dōngfāng () is one of the seven county-level cities of Hainan province, China."
4,52,0,G1,probe_response,yes
5,52,0,G1,probe_response,yes
6,52,0,G1,probe_response,yes
7,52,0,G1,probe_response,yes
8,5,1,G0,claim,"Question: Which actress who starred in ""About Last Night..."" endorsed the Werewolf Diet? Answer: The actress who starred in ""About Last Night..."" endorsed the Werewolf Diet is Angelina Jolie."
9,5,1,G0,knowledge,Celebrities that have endorsed the diet include Demi Moore and Madonna.
